# LimingjieNN · 黎明界高分双塔神经网络 & Path B 一键 Colab 流水线

本 Notebook 在 Google Colab 上一体化完成：
1. 挂载 Google Drive 与安装 `labyrinth-nn[train]`；
2. 运行含 PyTorch + ONNX Runtime `< 1e-4` 数值对齐在内的全套单元测试；
3. 使用离线锁定的 `master_cn.db` 拉起无头 `PCRCalculatorServer` 并运行 15 项国服口径对账 + 吞吐压测；
4. 提取客观特征目录 `feature_pack.json`；
5. 执行 Round 0 难度 5 分层自适应采样（支持 `/tmp` 本地分片 + 原子同步 Drive + `--resume` 断点续跑）；
6. 运行 `m0 -> m1 -> m2a -> m2b` 四档可归因消融实验并打印分切片指标；
7. 导出胜出配置的 `tower_a.onnx`、`tower_b.onnx`、`manifest.json` 与 `golden_io.json`；
8. 跑通 Path B 全局官方结算分回放对照基准。

In [ ]:
from pathlib import Path
import os, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running inside Colab; skipping drive.mount')

DRIVE_ROOT = Path('/content/drive/MyDrive/LimingjieNN')
REPO_SRC = DRIVE_ROOT / 'repo'          # Upload the LimingjieNN repo (without .venv/artifacts) here
REPO_DIR = Path('/content/LimingjieNN')  # Fast local working copy
CN_DB_PATH = DRIVE_ROOT / 'master_cn.db'  # Upload your locked CN master DB here
CN_DB_SHA256 = ''  # Optional expected SHA256 hex
CN_DB_VERSION = '202611131613'

if not (REPO_DIR / 'pyproject.toml').exists():
    if not (REPO_SRC / 'pyproject.toml').exists():
        raise FileNotFoundError(f'Repo not found: upload LimingjieNN to {REPO_SRC}')
    shutil.copytree(REPO_SRC, REPO_DIR, ignore=shutil.ignore_patterns('.venv', 'artifacts', '__pycache__', '*.egg-info', '.git'))
if not CN_DB_PATH.exists():
    raise FileNotFoundError(f'CN DB not found: upload master_cn.db to {CN_DB_PATH}')

CACHE_DIR = DRIVE_ROOT / 'cache'
DATA_DIR = DRIVE_ROOT / 'data'
RUNS_DIR = DRIVE_ROOT / 'runs'
EXPORT_DIR = DRIVE_ROOT / 'export_bundle'
for d in (CACHE_DIR, DATA_DIR, RUNS_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

%cd {REPO_DIR}
!pip install -q -e '.[train]'

import torch, onnxruntime as ort
print(f'CPU count={os.cpu_count()} | CUDA={torch.cuda.is_available()} | ORT={ort.__version__}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Verify all unit tests including Torch forward/backward, temperature calibration, and ONNX parity (< 1e-4)
!PYTHONPATH=src python -m unittest discover -s tests -v

In [ ]:
# Launch headless PCRCalculatorServer (offline locked CN DB) & run 15-check reconciliation + throughput probe
!chmod +x scripts/colab_setup_simulator.sh
cmd = (
    f'bash scripts/colab_setup_simulator.sh '
    f'--cache-dir "{CACHE_DIR}" '
    f'--db-path "{CN_DB_PATH}" '
    f'--db-version "{CN_DB_VERSION}" '
    f'--port 8080'
)
if CN_DB_SHA256:
    cmd += f' --db-sha256 "{CN_DB_SHA256}"'
get_ipython().system(cmd)

CATALOG_PATH = DATA_DIR / 'feature_pack.json'
!PYTHONPATH=src python -m labyrinth_nn.catalog.build_catalog --db "{CN_DB_PATH}" --out "{CATALOG_PATH}"
!PYTHONPATH=src python -m labyrinth_nn.data.probe_simulator \
    --catalog "{CATALOG_PATH}" \
    --profile configs/sim_profile_highscore_cn.json \
    --url http://127.0.0.1:8080 \
    --bench-count 64

In [ ]:
# 1) Dry-run distribution check (no simulator calls)
!PYTHONPATH=src python -m labyrinth_nn.data.sampler --catalog "{CATALOG_PATH}" --dry-run 2000

# 2) Fast smoke collection in local /tmp then sync to Drive (increase TARGET_BATTLES to 25000 or 100000 after probe)
import shutil
LOCAL_SAMPLES = Path('/tmp/round0_pilot.jsonl')
DRIVE_SAMPLES = DATA_DIR / 'round0_pilot.jsonl'
TARGET_BATTLES = 2000  # Pilot: 25000, Full: up to 150000

if DRIVE_SAMPLES.exists() and not LOCAL_SAMPLES.exists():
    shutil.copy2(DRIVE_SAMPLES, LOCAL_SAMPLES)

!PYTHONPATH=src python -m labyrinth_nn.data.sampler \
    --catalog "{CATALOG_PATH}" \
    --profile configs/sim_profile_highscore_cn.json \
    --sampling configs/sampling_round0.json \
    --url http://127.0.0.1:8080 \
    --out "{LOCAL_SAMPLES}" \
    --target-battles {TARGET_BATTLES} \
    --resume

shutil.copy2(LOCAL_SAMPLES, DRIVE_SAMPLES)
print('Synced samples to:', DRIVE_SAMPLES)

In [ ]:
# Run 4-step attributable ablation ladder: m0 -> m1 -> m2a -> m2b
ABLATION_DIR = RUNS_DIR / 'ablation_round0'
!PYTHONPATH=src python -m labyrinth_nn.train.train_critic \
    --configs m0 m1 m2a m2b \
    --seeds 42 \
    --amp \
    --catalog "{CATALOG_PATH}" \
    --samples "{DRIVE_SAMPLES}" \
    --out-dir "{ABLATION_DIR}"

print((ABLATION_DIR / 'ablation_summary.md').read_text(encoding='utf-8'))

In [ ]:
# Export winning ablation mode to ONNX + manifest.json + golden_io.json (validates PT vs ORT < 1e-4)
WINNING_MODE = 'm2b'  # Update based on ablation_summary.md
WINNING_CKPT = RUNS_DIR / 'ablation_round0' / WINNING_MODE / 'seed_42' / 'best.pt'
BUNDLE_DIR = EXPORT_DIR / WINNING_MODE

!PYTHONPATH=src python -m labyrinth_nn.export.export_onnx \
    --checkpoint "{WINNING_CKPT}" \
    --catalog "{CATALOG_PATH}" \
    --out-dir "{BUNDLE_DIR}"

import json
manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('Exported bundle:', BUNDLE_DIR, '| max_abs_diff:', manifest['validation']['max_abs_diff'])

In [ ]:
# Optional: Path B global official-score benchmark on recorded opening maps
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

from labyrinth_nn.catalog.schema import FeatureCatalog
from labyrinth_nn.env.run_env import PathBReplayPolicy, compare_policies, load_replays

REPLAY_PATH = DRIVE_ROOT / 'replays.jsonl'  # Optional recorded enter/map_list JSONL
if REPLAY_PATH.exists():
    catalog = FeatureCatalog.load(CATALOG_PATH)
    episodes = load_replays(REPLAY_PATH)
    reports = compare_policies(
        episodes,
        path_b=PathBReplayPolicy(target_score=53000),
        settings=catalog.score_settings,
        catalog=catalog,
        repeats=5,
    )
    for name, rep in reports.items():
        print(name, rep)
else:
    print(f'Place recorded opening maps at {REPLAY_PATH} to run Path B replay comparison.')